# Subfase I: Ingesta, limpieza y control

Objetivo de la subfase:
- Cargar el dataset crudo transaccional de BVG
- Estandarizar tipos de datos y calidad
- Filtrar emisores objetivo para el pipeline
- Exportar dataset limpio para la siguiente subfase
- Preservar orden temporal por emisor

In [9]:
import pandas as pd
import seaborn as sns

# Variables fijas y rutas
from bvg_core.config import (
  DATASET_RAW_PATH,
  DATASET_PROCESSED_PATH,
  COMPANIES,
  EMISOR_COL,
  FECHA_COL,
  PRECIO_COL,
  ACCIONES_COL,
  VALOR_EFECTO_COL,
  VALOR_NOMINAL_COL,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 250)
sns.set_theme(style='whitegrid')

In [10]:
# Abrir archivo con pandas
df = pd.read_csv(DATASET_RAW_PATH, sep=';')

print(f'Filas cargadas: {len(df):,}')
print(f'Columnas cargadas: {len(df.columns)}')

Filas cargadas: 28,592
Columnas cargadas: 11


In [11]:
# Inspeccion general del dataset
print('Shape del dataset:', df.shape)
print('\nColumnas disponibles:')
display(df.columns.tolist())
print('\nTipos de datos:')
display(df.dtypes)

print('\nPrimeras filas:')
display(df.head())
print('\nNulos por columna:')
display(df.isna().sum())

Shape del dataset: (28592, 11)

Columnas disponibles:


['FECHA NEGOCIACIÓN',
 'TÍTULO',
 'EMISOR',
 'NÚMERO DE ACCIONES',
 'V. NOM. UNITARIO',
 'PRECIO',
 'VALOR NOMINAL',
 'VALOR EFECTO',
 'CASA COMPRADORA',
 'CASA VENDEDORA',
 'BOLSA']


Tipos de datos:


FECHA NEGOCIACIÓN         str
TÍTULO                    str
EMISOR                    str
NÚMERO DE ACCIONES        str
V. NOM. UNITARIO          str
PRECIO                float64
VALOR NOMINAL             str
VALOR EFECTO              str
CASA COMPRADORA           str
CASA VENDEDORA            str
BOLSA                     str
dtype: object


Primeras filas:


,FECHA NEGOCIACIÓN,TÍTULO,EMISOR,NÚMERO DE ACCIONES,V. NOM. UNITARIO,PRECIO,VALOR NOMINAL,VALOR EFECTO,CASA COMPRADORA,CASA VENDEDORA,BOLSA
0,02/01/2019,ACCIONES,CORPORACION FAVORITA C.A.,1.000.00,1.00,2.45,1.000.00,2.450.00,MERCAPITAL,ORION,BVG
1,02/01/2019,ACCIONES,CORPORACION FAVORITA C.A.,821.00,1.00,2.44,821.00,2.003.24,PICAVAL,PICAVAL,BVG
2,02/01/2019,ACCIONES,SURPAPELCORP S.A.,90.00,1.00,4.25,90.00,382.50,ACCIONES Y VALORES,ACCIONES Y VALORES,BVG
3,02/01/2019,ACCIONES,CORPORACION FAVORITA C.A.,2.027.00,1.00,2.44,2.027.00,4.945.88,SANTA FE,PICAVAL,BVG
4,02/01/2019,ACCIONES,CORPORACION FAVORITA C.A.,189.00,1.00,2.44,189.00,461.16,SILVERCROSS,PICAVAL,BVG



Nulos por columna:


FECHA NEGOCIACIÓN     0
TÍTULO                0
EMISOR                0
NÚMERO DE ACCIONES    0
V. NOM. UNITARIO      0
PRECIO                0
VALOR NOMINAL         0
VALOR EFECTO          0
CASA COMPRADORA       0
CASA VENDEDORA        0
BOLSA                 0
dtype: int64

In [12]:
# Filtrado por empresas objetivo
df_obj = df[df[EMISOR_COL].isin(COMPANIES)].copy()

print('Shape del dataset filtrado:', df_obj.shape)
print('Registros por empresa:')
display(df_obj[EMISOR_COL].value_counts())

Shape del dataset filtrado: (18188, 11)
Registros por empresa:


EMISOR
CORPORACION FAVORITA C.A.    14706
BANCO GUAYAQUIL S.A.          3482
Name: count, dtype: int64

In [13]:
# Conversion y limpieza de tipos
n_inicial = len(df_obj)

df_obj[FECHA_COL] = pd.to_datetime(
    df_obj[FECHA_COL],
    format='%d/%m/%Y',
    errors='coerce',
)

def limpiar_numero_serie(serie: pd.Series) -> pd.Series:
    s = serie.astype('string').str.strip()
    s = s.str.replace(r'[^0-9\.\-]', '', regex=True)
    # Elimina todos los puntos excepto el ultimo para soportar separadores de miles inconsistentes
    s = s.str.replace(r'\.(?=.*\.)', '', regex=True)
    return pd.to_numeric(s, errors='coerce')

df_obj[PRECIO_COL] = pd.to_numeric(df_obj[PRECIO_COL], errors='coerce')
for col in [ACCIONES_COL, VALOR_EFECTO_COL, VALOR_NOMINAL_COL]:
    df_obj[col] = limpiar_numero_serie(df_obj[col])

n_fecha_invalida = int(df_obj[FECHA_COL].isna().sum())
n_precio_invalido = int(df_obj[PRECIO_COL].isna().sum())

df_obj = df_obj.dropna(subset=[FECHA_COL, PRECIO_COL]).copy()
n_final = len(df_obj)

print(f'Fechas no convertibles (NaT): {n_fecha_invalida}')
print(f'Precios no convertibles (NaN): {n_precio_invalido}')
print(f'Registros iniciales filtrados: {n_inicial:,}')
print(f'Registros finales luego de limpieza: {n_final:,}')
print(f'Registros removidos por calidad: {n_inicial - n_final:,}')

display(df_obj.dtypes)

Fechas no convertibles (NaT): 0
Precios no convertibles (NaN): 0
Registros iniciales filtrados: 18,188
Registros finales luego de limpieza: 18,188
Registros removidos por calidad: 0


FECHA NEGOCIACIÓN     datetime64[us]
TÍTULO                           str
EMISOR                           str
NÚMERO DE ACCIONES           Float64
V. NOM. UNITARIO                 str
PRECIO                       float64
VALOR NOMINAL                Float64
VALOR EFECTO                 Float64
CASA COMPRADORA                  str
CASA VENDEDORA                   str
BOLSA                            str
dtype: object

In [14]:
# Ordenamiento temporal
df_obj = df_obj.sort_values([EMISOR_COL, FECHA_COL]).reset_index(drop=True)
display(df_obj.head())
display(df_obj.tail())

,FECHA NEGOCIACIÓN,TÍTULO,EMISOR,NÚMERO DE ACCIONES,V. NOM. UNITARIO,PRECIO,VALOR NOMINAL,VALOR EFECTO,CASA COMPRADORA,CASA VENDEDORA,BOLSA
0,2019-01-02,ACCIONES,BANCO GUAYAQUIL S.A.,2000.0,1.00,0.96,2000.0,1920.0,PLUSBURSÁTIL,SANTA FE,BVG
1,2019-01-03,ACCIONES,BANCO GUAYAQUIL S.A.,11069.0,1.00,0.96,11069.0,10626.24,SILVERCROSS,SILVERCROSS,BVG
2,2019-01-03,ACCIONES,BANCO GUAYAQUIL S.A.,20035.0,1.00,0.96,20035.0,19233.6,SILVERCROSS,SILVERCROSS,BVG
3,2019-01-14,ACCIONES,BANCO GUAYAQUIL S.A.,1978.0,1.00,0.96,1978.0,1898.88,SANTA FE,SANTA FE,BVG
4,2019-01-14,ACCIONES,BANCO GUAYAQUIL S.A.,13022.0,1.00,0.95,13022.0,12370.9,SANTA FE,ORION,BVG


,FECHA NEGOCIACIÓN,TÍTULO,EMISOR,NÚMERO DE ACCIONES,V. NOM. UNITARIO,PRECIO,VALOR NOMINAL,VALOR EFECTO,CASA COMPRADORA,CASA VENDEDORA,BOLSA
18183,2026-03-25,ACCIONES,CORPORACION FAVORITA C.A.,1191.0,1,2.0,1191.0,2382.0,ECUABURSÁTIL,MERCAPITAL,BVQ
18184,2026-03-25,ACCIONES,CORPORACION FAVORITA C.A.,4000.0,1,2.0,4000.0,8000.0,ECUABURSÁTIL,ECUABURSÁTIL,BVQ
18185,2026-03-25,ACCIONES,CORPORACION FAVORITA C.A.,2541.0,1,2.0,2541.0,5082.0,ECUABURSÁTIL,METROVALORES,BVQ
18186,2026-03-25,ACCIONES,CORPORACION FAVORITA C.A.,316.0,1,2.0,316.0,632.0,ECUABURSÁTIL,SANTA FE,BVG
18187,2026-03-25,ACCIONES,CORPORACION FAVORITA C.A.,3201.0,1,2.0,3201.0,6402.0,ECUABURSÁTIL,SANTA FE,BVG


In [15]:
# Rango de fechas y numero total de registros por empresa
resumen_empresas = (
    df_obj.groupby(EMISOR_COL)
    .agg(
        total_registros=(EMISOR_COL, 'size'),
        fecha_min=(FECHA_COL, 'min'),
        fecha_max=(FECHA_COL, 'max')
    )
    .sort_values('total_registros', ascending=False)
)

display(resumen_empresas)

,total_registros,fecha_min,fecha_max
EMISOR,,,
CORPORACION FAVORITA C.A.,14706,2019-01-02,2026-03-25
BANCO GUAYAQUIL S.A.,3482,2019-01-02,2026-03-25


In [16]:
# Guardar dataset limpio para subfase 1.2
DATASET_PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df_obj.to_csv(DATASET_PROCESSED_PATH, index=False)
print(f'Dataset limpio exportado: {DATASET_PROCESSED_PATH.as_posix()}')

Dataset limpio exportado: C:/Users/leynd/Desktop/Tesis/implementaciones/desarrollov3/data/processed/BVG_Acciones_limpio.csv


## Conclusiones de subfase

### 1. Carga
- Se cargó y perfiló el dataset crudo de BVG.
- Se filtraron los emisores objetivo para el flujo de tesis.
- Se trazó la cantidad de registros removidos por calidad de fecha/precio.

### 2. Salida de la subfase
- Archivo limpio y ordenado temporalmente para la subfase de integridad temporal: `data/processed/BVG_Acciones_limpio.csv`.